# 문장 임베딩과 유사도 — 수강생 실습본

원본: `src/raw/Chapter01/03-rag/02_embeddings.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-4-rag-theory-and-core-modules.pdf`

**학습 목표:** 문장 벡터 → 코사인 유사도 → 가까운 문장 선택의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 세 문장과 질문을 임베딩 API로 보낸다. 사내 또는 OpenAI 임베딩 설정과 인증을 확인한다.



## 흐름을 먼저 읽기

문장 벡터 → 코사인 유사도 → 가까운 문장 선택

핵심은 문서 준비·검색·답변 단계의 경계이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 문장 벡터 → 코사인 유사도 → 가까운 문장 선택. 강아지·개 문장이 주식 문장보다 가까운 이유는? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 세 문장과 질문을 임베딩 API로 보낸다. 사내 또는 OpenAI 임베딩 설정과 인증을 확인한다.



In [ ]:
"""
임베딩(Embedding): 텍스트를 숫자 벡터로 변환하기

핵심 개념:
  - 임베딩 = 텍스트의 '의미'를 숫자 벡터로 표현
  - 의미가 비슷한 텍스트는 벡터 공간에서 가까이 위치
  - RAG에서 '관련 문서 검색'은 이 벡터 거리를 계산하는 것

실행:
  python 03-rag/02_embeddings.py
"""
import os

import numpy as np
from dotenv import load_dotenv
from config.embeddings import get_embeddings
from config.samsung import embedding_config

load_dotenv()

import requests
from langchain_core.embeddings import Embeddings

BATCH   = 10  # API 한 번에 처리 가능한 최대 텍스트 수

class SamsungEmbeddings(Embeddings):

    def embed_documents(self, texts):
        # BATCH 개씩 나눠서 API 호출, 결과를 하나로 합침
        result = []
        for i in range(0, len(texts), BATCH):
            # 사내 임베딩 주소와 인증 헤더는 config/samsung.py에 모아 두었다.
            config = embedding_config()
            res = requests.post(
                config['url'],
                json={'input': texts[i:i + BATCH]},
                headers=config['headers'],
                timeout=30,
            )
            res.raise_for_status()
            result.extend(item['embedding'] for item in res.json()['data'])
        return result

    def embed_query(self, text):
        # 질문 하나를 임베딩할 때는 embed_documents 재활용
        return self.embed_documents([text])[0]

provider = os.getenv('EMBED_PROVIDER', 'samsung').strip().lower()
if provider == 'samsung':
    embedding_config()  # 설정이 비어 있으면 여기서 멈춘다.
    embeddings = SamsungEmbeddings()
else:
    # 사외에서는 같은 유사도 실습을 OpenAI 임베딩으로 실행한다.
    embeddings = get_embeddings()

sentences = [
    '강아지가 공원에서 뛰어놀고 있다.',
    '개가 잔디밭에서 달리고 있다.',
    '오늘 주식시장이 크게 하락했다.',
]

# embed_documents(): 텍스트 리스트 → 벡터 리스트
vectors = embeddings.embed_documents(sentences)

print('=== 벡터 기본 정보 ===')
print(f'문장 수    : {len(vectors)}')
print(f'벡터 차원  : {len(vectors[0])}')
print(f'앞 5개 값  : {[v[:5] for v in vectors]}')
print()


# ── 코사인 유사도 ─────────────────────────────────────────────────────────────
# 코사인 유사도: 1에 가까울수록 의미가 유사, 0에 가까울수록 관련 없음
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

v = [np.array(vec) for vec in vectors]
print('=== 문장 간 유사도 ===')
print(f'[강아지] vs [개]   : {cosine_similarity(v[0], v[1]):.4f}')   # 높아야 함
print(f'[강아지] vs [주식] : {cosine_similarity(v[0], v[2]):.4f}')   # 낮아야 함
print()

# embed_query(): 질문 하나를 벡터로 변환 (검색 시 사용)
query_vec = np.array(embeddings.embed_query('애완동물이 뛰노는 모습'))
scores = [cosine_similarity(query_vec, v_i) for v_i in v]
best = scores.index(max(scores))
print(f'질문: "애완동물이 뛰노는 모습"')
print(f'가장 유사한 문장: "{sentences[best]}" (유사도: {scores[best]:.4f})')
print()


# ── 실습: SamsungEmbeddings 구현 ──────────────────────────────────────────────
# 삼성 사내 임베딩 API는 LangChain과 직접 호환되지 않는다.
# langchain_core.embeddings.Embeddings 를 상속해 아래 두 메서드만 구현하면
# vectorstore, retriever 등 LangChain의 모든 컴포넌트에서 그대로 사용 가능하다.
#
# 아래 주석을 해제하고 SamsungEmbeddings 를 완성한 뒤,
# embeddings = SamsungEmbeddings() 로 교체해 위 유사도 계산을 다시 실행해 보자.





## 결과를 검토하기

- 예상 동작: 벡터 차원과 문장 사이 유사도, 질문과 가장 가까운 문장을 출력한다.
- 확인 질문: 강아지·개 문장이 주식 문장보다 가까운 이유는?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
